In [9]:
from pysentimiento import create_analyzer
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
analyzer = create_analyzer(task="sentiment", lang="en")
emotion_analyzer = create_analyzer(task="emotion", lang="en")
hate_speech_analyzer = create_analyzer(task="hate_speech", lang="en")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
df = pd.read_csv('combined_years.csv', index_col=0)

In [11]:
def sentiment_analysis(row):
    result = analyzer.predict(row['llm_text'])
    main_output = result.output
    pos_prob = result.probas['POS']
    neg_prob = result.probas['NEG']
    neu_prob = result.probas['NEU']
    return main_output, pos_prob, neg_prob, neu_prob


sentiment_new_columns = df.progress_apply(
    sentiment_analysis, axis=1, result_type='expand')
sentiment_new_columns.columns = ['sent_label', 'sent_pos_prob',
                                 'sent_neg_prob', 'sent_neu_prob']

  0%|          | 0/83054 [00:00<?, ?it/s]

In [12]:
def emotion_analysis(row):
    result = emotion_analyzer.predict(row['llm_text'])
    main_output = result.output
    others_prob = result.probas['others']
    joy_prob = result.probas['joy']
    sadness_prob = result.probas['sadness']
    anger_prob = result.probas['anger']
    surprise_prob = result.probas['surprise']
    disgust_prob = result.probas['disgust']
    fear_prob = result.probas['fear']
    return main_output, others_prob, joy_prob, sadness_prob, anger_prob, surprise_prob, disgust_prob, fear_prob


emotion_new_columns = df.progress_apply(
    emotion_analysis, axis=1, result_type='expand')
emotion_new_columns.columns = ['emo_label', 'emo_others_prob', 'emo_joy_prob', 'emo_sadness_prob',
                               'emo_anger_prob', 'emo_surprise_prob', 'emo_disgust_prob', 'emo_fear_prob']

  0%|          | 0/83054 [00:00<?, ?it/s]

In [13]:
def hate_speech_analysis(row):
    result = hate_speech_analyzer.predict(row['llm_text'])
    hateful_prob = result.probas['hateful']
    targeted_prob = result.probas['targeted']
    aggressive_prob = result.probas['aggressive']
    return hateful_prob, targeted_prob, aggressive_prob


hate_new_columns = df.progress_apply(
    hate_speech_analysis, axis=1, result_type='expand')
hate_new_columns.columns = ['speech_hateful_prob',
                            'speech_targeted_prob', 'speech_aggressive_prob']

  0%|          | 0/83054 [00:00<?, ?it/s]

In [14]:
df = df.join(hate_new_columns)
df = df.join(emotion_new_columns)
df = df.join(sentiment_new_columns)

In [16]:
df.to_csv('data_v1.csv')